# GC-SUAE — WHISPERS rerun
Run cells top to bottom. **Runtime → Change runtime type → T4 GPU** before starting.

Everything is written to Drive, so a disconnect loses nothing that has finished; just re-run from the top (finished steps are skipped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/Lunar_dataset"   # your data folder
import os; print(sorted(os.listdir(D)))

## 1. Code

In [ ]:
%cd /content
!rm -rf Lunarspecnet && git clone https://github.com/Vaibhavtripathi7/Lunarspecnet.git
%cd /content/Lunarspecnet
!pip install -q -e . 2>&1 | tail -1
!git log --oneline -1

In [ ]:
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — fix runtime type before continuing")

## 2. Kaguya MI FeO (wt%) — crop the scene window from the USGS global mosaic
Reads ~6 GB of rows over HTTP, keeps ~50 MB. 5–15 min. Skipped if the file already exists.

In [ ]:
import os, rasterio, numpy as np
from rasterio.windows import from_bounds
os.environ.update({"GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES",
                   "CPL_VSIL_CURL_CHUNK_SIZE": "16777216",
                   "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"})
URL = "/vsicurl/https://planetarymaps.usgs.gov/mosaic/Lunar_MI_mineral_maps/Lunar_Kaguya_MIMap_MineralDeconv_FeOWeightPercent_50N50S.tif"
OUT = f"{D}/kaguya_feo_scene.tif"
west, south, east, north = 15.0, -41.0, 18.0, -24.0    # generous box around the IIRS strip

if os.path.exists(OUT):
    print("already exists:", OUT)
else:
    with rasterio.open(URL) as src:
        win = from_bounds(west, south, east, north, src.transform)
        data = src.read(1, window=win)
        meta = src.meta.copy()
        meta.update(height=data.shape[0], width=data.shape[1], driver="GTiff",
                    transform=src.window_transform(win), compress="deflate")
        with rasterio.open(OUT, "w", **meta) as dst:
            dst.write(data, 1)
    print("saved", OUT)

with rasterio.open(OUT) as s:
    a = s.read(1); v = a[a != s.nodata]
    print("shape", a.shape, "| valid %.1f%%" % (100*v.size/a.size))
    print("FeO wt%%: min %.2f  2nd %.2f  median %.2f  98th %.2f  max %.2f"
          % (v.min(), np.percentile(v,2), np.median(v), np.percentile(v,98), v.max()))

## 3. Register DEM + FeO onto the IIRS pixel grid
Needs `ch2_iir_..._d_loc_d18_ard.img` and `.hdr` in the Drive folder (from the PRADAN bundle, `data/derived/20200703/`).

Expected: DEM valid ≈ 75 %, `corr 0.9788`; FeO valid ≈ 100 %.

In [ ]:
!python scripts/register_aux.py \
    --loc {D}/ch2_iir_ndi_20200703T1535029218_d_loc_d18_ard.img \
    --dem {D}/ch2_tmc_ndn_20200703T1535027868_d_dtm_d18.tif \
    --feo {D}/kaguya_feo_scene.tif \
    --output_dir {D}/registered

## 4. Point the config at the data

In [ ]:
import re, pathlib
p = pathlib.Path("configs/ablation.yaml"); s = p.read_text()
s = re.sub(r'iirs_hdr: ".*"', f'iirs_hdr: "{D}/data.hdr"', s)
s = re.sub(r'iirs_qub: ".*"', f'iirs_qub: "{D}/data.qub"', s)
s = re.sub(r'dem_path: ".*"', f'dem_path: "{D}/registered/aligned_tmc2_dem.tif"', s)
s = re.sub(r'feo_path: ".*"', f'feo_path: "{D}/registered/aligned_elemental_map.tif"', s)
p.write_text(s)
print(s[:1200])

## 5. Train all 7 models × 3 seeds (~3 h per seed on T4)
Attention models first so the key comparison lands earliest. Finished models are skipped on re-run.

In [ ]:
MODELS = "pooled_attn_fusion,gcsuae_no_tagcl,gcsuae,late_fusion,early_fusion,unimodal_3d,unimodal_2d"
for seed in [42, 7, 123]:
    !python scripts/run_ablation.py --config configs/ablation.yaml --seed {seed} \
        --output-dir {D}/ablation_seeds/seed{seed} --models {MODELS}

## 6. Aggregate (run any time; uses whatever seeds have finished)

In [ ]:
!python scripts/aggregate_seeds.py --root {D}/ablation_seeds --latex \
    --compare PooledAttnFusion GC_SUAE_NoTAGCL